# 🎬 שרת ההורדות החינמי ל-SaveBridge

מריץ שרת קטן ב-Colab שמוריד סרטוני YouTube עם `yt-dlp`+`ffmpeg`, וחושף אותו
בכתובת אינטרנט זמנית. מדביקים את הכתובת בהגדרות התוסף — וזהו, בלי פרוקסי בתשלום.

**איך משתמשים:**
1. תפריט **Runtime → Run all** (או ▶ תא-תא).
2. בסוף מודפסת **כתובת השרת** (`https://....trycloudflare.com`).
3. פותחים את הגדרות התוסף ב-Chrome, מדביקים את הכתובת בשדה "כתובת השרת", לוחצים **שמור וחבר**.
4. נכנסים ל-YouTube ולוחצים על כפתור ההורדה.

> ⚠️ השאר את התא האחרון **רץ** — כשעוצרים אותו, השרת נסגר והכתובת מתבטלת.
> ל-Colab יש מגבלת זמן; בפעם הבאה מריצים שוב ומדביקים את הכתובת החדשה.


## 1. התקנת רכיבים (yt-dlp, ffmpeg, cloudflared) + הורדת קוד השרת

In [ ]:
# עדכן BRANCH ל-main אחרי שה-PR מתמזג.
REPO   = 'https://github.com/g47ben-lang/zmani'
BRANCH = 'claude/youtube-plugin-no-proxy-ujrtjt'

import os
!pip -q install 'fastapi' 'uvicorn[standard]' 'yt-dlp' 'python-multipart'
!apt-get -qq install -y ffmpeg > /dev/null
if not os.path.isdir('zmani'):
    !git clone -q -b "$BRANCH" "$REPO"
else:
    !cd zmani && git pull -q

# cloudflared (חושף את השרת המקומי לכתובת אינטרנט)
if not os.path.exists('cloudflared'):
    import urllib.request
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        'cloudflared')
    os.chmod('cloudflared', 0o755)
print('✅ הכל מוכן')


## 2. (רשות) סיסמה לשרת

הכתובת של Colab אקראית וקשה לניחוש, אז אפשר להשאיר ריק. אם בכל זאת רוצים
סיסמה — כתבו אותה כאן, והדביקו אותה גם בשדה "טוקן" בהגדרות התוסף.

In [ ]:
import os
SB_TOKEN = ''   # למשל 'my-secret-123'. ריק = פתוח (מומלץ לכתובת אקראית).
os.environ['SB_TOKEN'] = SB_TOKEN
print('סיסמה:', SB_TOKEN or '(ללא — פתוח)')


## 3. הפעלת השרת + קבלת הכתובת  ← השאר תא זה רץ

In [ ]:
import subprocess, sys, os, time, re

os.environ['PORT'] = '9797'

# מפעיל את שרת ההורדות
server = subprocess.Popen(
    [sys.executable, 'app.py'],
    cwd='zmani/youtube-extension/server',
    env={**os.environ},
)
time.sleep(4)

# פותח מנהרה לכתובת אינטרנט
tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:9797', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public = None
for line in tunnel.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        public = m.group(0)
        break

print('\n' + '=' * 64)
print('  📋 כתובת השרת — הדבק בהגדרות התוסף בשדה "כתובת השרת":')
print('     ', public or '(לא נמצאה כתובת — הרץ שוב את התא)')
print('=' * 64)
print('\n⏳ השאר את התא הזה רץ. עצירה = השרת נסגר.\n')

try:
    for _ in tunnel.stdout:
        pass
except KeyboardInterrupt:
    tunnel.terminate(); server.terminate()
    print('נעצר.')
